In [ ]:
# 1. Setup de caminhos locais
import os
import subprocess
from datetime import datetime
from pathlib import Path

from src.io import paths
from src.config import loader as cfg
from src.validation import merges
from src.utils.logger import log_result

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])
CONFIG = cfg.load_config()

NB_NAME = "16_models_tfidf_baselines"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("BASE_PATH:", BASE_PATH)
print("PROC_PATH:", PROC_PATH)


In [ ]:
# 2. Bibliotecas e caminhos esperados
import json
import numpy as np
import pandas as pd
from pathlib import Path

from scipy.sparse import load_npz
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import TimeSeriesSplit

import plotly.graph_objects as go

paths = {
    "matrix": os.path.join(PROC_PATH, "tfidf_daily_matrix.npz"),
    "index": os.path.join(PROC_PATH, "tfidf_daily_index.csv"),
    "labels": os.path.join(PROC_PATH, "labels_y_daily.csv"),
}

for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Arquivo obrigatório não encontrado: {path}. Rode o notebook 15 antes.")

print(json.dumps(paths, indent=2, ensure_ascii=False))

In [ ]:
# 3. Carregar matriz TF-IDF diária e labels alinhados
X_all = load_npz(paths["matrix"]).tocsr()
idx = pd.read_csv(paths["index"], parse_dates=["day"])
y_daily = pd.read_csv(paths["labels"], parse_dates=["day"])
labels = y_daily.copy()

base = idx.merge(y_daily, how="left", on="day")
mask = base["y"].notna()
if mask.sum() == 0:
    raise ValueError("Arquivo labels_y_daily.csv não possui coluna 'y' preenchida.")

X = X_all[mask.values]
df_model = base.loc[mask].reset_index(drop=True).copy()
y = df_model["y"].astype(int).to_numpy()

if "ret_next" not in df_model.columns:
    raise ValueError("labels_y_daily.csv precisa ter a coluna 'ret_next'.")

price_cols = [c for c in ["close", "adj_close", "price"] if c in df_model.columns]
price_col = price_cols[0] if price_cols else None
if price_col is None:
    df_model["close"] = np.nan
    price_col = "close"

print(f"Amostras com rótulo: {len(df_model)} | shape matriz: {X.shape}")
display(df_model.head())

In [ ]:
# Checar interseção entre TF-IDF e labels
merges.check_intersection(idx, labels, col_left="day", col_right="day", min_days=90)

In [ ]:
# 4. Funções auxiliares (MDA, bootstrap, etc)
from typing import Callable, Dict


def mean_directional_accuracy(y_true: np.ndarray, proba: np.ndarray, threshold: float = 0.5) -> float:
    if len(y_true) == 0:
        return np.nan
    preds = (proba >= threshold).astype(int)
    return float((preds == y_true).mean())


def bootstrap_ci(
    y_true: np.ndarray,
    scores: np.ndarray,
    scorer: Callable[[np.ndarray, np.ndarray], float],
    n_boot: int = 1000,
    seed: int = 42,
    requires_two_classes: bool = False,
) -> Dict[str, float]:
    rng = np.random.default_rng(seed)
    idx = np.arange(len(y_true))
    stats = []
    for _ in range(n_boot):
        sample_idx = rng.choice(idx, size=len(idx), replace=True)
        y_sample = y_true[sample_idx]
        if requires_two_classes and np.unique(y_sample).size < 2:
            continue
        try:
            stats.append(float(scorer(y_sample, scores[sample_idx])))
        except ValueError:
            continue
    if not stats:
        return {"mean": np.nan, "low": np.nan, "high": np.nan, "n": 0}
    return {
        "mean": float(np.mean(stats)),
        "low": float(np.percentile(stats, 2.5)),
        "high": float(np.percentile(stats, 97.5)),
        "n": len(stats),
    }

In [ ]:
# 5. Configurar modelos baseline + walk-forward
MODELS = {
    "logreg_l2": LogisticRegression(
        solver="saga",
        penalty="l2",
        C=1.0,
        max_iter=2000,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
    "rf_200": RandomForestClassifier(
        n_estimators=200,
        max_depth=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    ),
}

n_splits = min(4, max(2, len(df_model) - 1))
tscv = TimeSeriesSplit(n_splits=n_splits)

summary_rows = []
oof_records = []
fold_records = []
model_preds = {}

for model_name, base_model in MODELS.items():
    preds = np.full(len(df_model), np.nan, dtype=float)
    for fold_id, (train_idx, test_idx) in enumerate(tscv.split(X)):
        y_train = y[train_idx]
        if np.unique(y_train).size < 2:
            print(f"?? {model_name} - fold {fold_id} ignorado (uma classe só no treino)")
            continue
        model = clone(base_model)
        model.fit(X[train_idx], y_train)
        proba = model.predict_proba(X[test_idx])[:, 1]
        preds[test_idx] = proba

        y_test = y[test_idx]
        auc_fold = roc_auc_score(y_test, proba) if np.unique(y_test).size > 1 else np.nan
        mda_fold = mean_directional_accuracy(y_test, proba)
        fold_records.append({
            "model": model_name,
            "fold": int(fold_id),
            "train_start": str(df_model.loc[train_idx[0], "day"].date()),
            "train_end": str(df_model.loc[train_idx[-1], "day"].date()),
            "test_start": str(df_model.loc[test_idx[0], "day"].date()),
            "test_end": str(df_model.loc[test_idx[-1], "day"].date()),
            "n_train": int(len(train_idx)),
            "n_test": int(len(test_idx)),
            "auc": float(auc_fold) if not np.isnan(auc_fold) else np.nan,
            "mda": float(mda_fold) if not np.isnan(mda_fold) else np.nan,
        })

        for local_idx, row_idx in enumerate(test_idx):
            row = df_model.loc[row_idx]
            oof_records.append({
                "model": model_name,
                "fold": int(fold_id),
                "row_id": int(row_idx),
                "day": row["day"],
                "y": int(row["y"]),
                "ret_next": float(row["ret_next"]),
                "close": float(row[price_col]) if price_col in row else np.nan,
                "proba": float(proba[local_idx]),
            })

    valid = ~np.isnan(preds)
    if valid.sum() == 0:
        print(f"Sem previsões válidas para {model_name}")
        continue
    y_valid = y[valid]
    p_valid = preds[valid]

    auc_full = roc_auc_score(y_valid, p_valid) if np.unique(y_valid).size > 1 else np.nan
    mda_full = mean_directional_accuracy(y_valid, p_valid)

    auc_ci = bootstrap_ci(y_valid, p_valid, roc_auc_score, requires_two_classes=True)
    mda_ci = bootstrap_ci(y_valid, p_valid, mean_directional_accuracy)

    summary_rows.append({
        "model": model_name,
        "auc": float(auc_full) if not np.isnan(auc_full) else np.nan,
        "auc_low": auc_ci["low"],
        "auc_high": auc_ci["high"],
        "mda": float(mda_full) if not np.isnan(mda_full) else np.nan,
        "mda_low": mda_ci["low"],
        "mda_high": mda_ci["high"],
        "coverage": float(valid.mean()),
        "n_obs": int(valid.sum()),
    })

    model_preds[model_name] = p_valid

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

In [ ]:
# Registrar métricas agregadas
for row in summary_rows:
    metrics = {
        "auc": float(row.get("auc")) if row.get("auc") is not None else None,
        "mda": float(row.get("mda")) if row.get("mda") is not None else None,
    }
    extra = {
        "dataset": "tfidf_daily",
        "coverage": float(row.get("coverage", 0)),
        "n_obs": int(row.get("n_obs", 0)),
        "auc_ci": (row.get("auc_low"), row.get("auc_high")),
        "mda_ci": (row.get("mda_low"), row.get("mda_high")),
    }
    log_result(model_name=row["model"], dataset_name="tfidf_daily", metrics=metrics, extra=extra)


In [ ]:
# 6. Persistir métricas + OOF
results_payload = {
    "notebook": NB_NAME,
    "generated_at": STAMP,
    "n_samples": int(len(df_model)),
    "models": {},
}
for row in summary_rows:
    name = row["model"]
    results_payload["models"][name] = {
        "auc": {"value": row["auc"], "low": row["auc_low"], "high": row["auc_high"]},
        "mda": {"value": row["mda"], "low": row["mda_low"], "high": row["mda_high"]},
    }

results_file = os.path.join(PROC_PATH, "results_16_models_tfidf.json")
with open(results_file, "w", encoding="utf-8") as f:
    json.dump(results_payload, f, indent=2, ensure_ascii=False)

fold_df = pd.DataFrame(fold_records)
fold_file = os.path.join(PROC_PATH, "16_cv_folds_metrics.csv")
fold_df.to_csv(fold_file, index=False, encoding="utf-8")

oof_df = pd.DataFrame(oof_records)
oof_df.sort_values(["model", "day"], inplace=True)
oof_file = os.path.join(PROC_PATH, "16_oof_predictions.csv")
oof_df.to_csv(oof_file, index=False, encoding="utf-8")

print("Resultados salvos:")
print(results_file)
print(fold_file)
print(oof_file)

In [ ]:
# 7. Curvas ROC e gráfico de ganho acumulado
if not oof_records:
    raise RuntimeError("Sem previsões para plotar.")

oof_df = pd.DataFrame(oof_records)
roc_fig = go.Figure()
for model_name in oof_df["model"].unique():
    sub = oof_df[oof_df["model"] == model_name].dropna(subset=["proba"])
    if sub["y"].nunique() < 2:
        continue
    fpr, tpr, _ = roc_curve(sub["y"], sub["proba"])
    roc_auc = roc_auc_score(sub["y"], sub["proba"]) if sub["y"].nunique() > 1 else np.nan
    roc_fig.add_trace(
        go.Scatter(x=fpr, y=tpr, mode="lines", name=f"{model_name} (AUC={roc_auc:.3f})")
    )
roc_fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Baseline", line=dict(dash="dash")))
roc_fig.update_layout(title="Curvas ROC (OOF)", xaxis_title="FPR", yaxis_title="TPR")
roc_file = os.path.join(PROC_PATH, "16_roc_curves.html")
roc_fig.write_html(roc_file)
display(roc_fig)

lift_fig = go.Figure()
for model_name in oof_df["model"].unique():
    sub = oof_df[oof_df["model"] == model_name].dropna(subset=["proba"])
    if sub.empty or sub["y"].sum() == 0:
        continue
    ordered = sub.sort_values("proba", ascending=False).reset_index(drop=True)
    ordered["pct_obs"] = (ordered.index + 1) / len(ordered)
    ordered["cum_target"] = ordered["y"].cumsum() / ordered["y"].sum()
    lift_fig.add_trace(
        go.Scatter(
            x=ordered["pct_obs"],
            y=ordered["cum_target"],
            mode="lines",
            name=model_name,
        )
    )
lift_fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Aleatório", line=dict(dash="dot")))
lift_fig.update_layout(title="Ganho acumulado", xaxis_title="% observações", yaxis_title="% positivos acumulados")
lift_file = os.path.join(PROC_PATH, "16_gain_curves.html")
lift_fig.write_html(lift_file)
display(lift_fig)

print("Gráficos salvos:")
print(roc_file)
print(lift_file)

In [ ]:
# 8. Resumo final
from IPython.display import Markdown

if not summary_rows:
    raise RuntimeError("Nenhum modelo avaliado.")

md = """**16_models_tfidf_baselines concluído ✅**
- Amostras etiquetadas: **{len(df_model)}**
- Splits temporais: **{n_splits}**
- Artefatos atualizados:
  - data_processed/results_16_models_tfidf.json
  - data_processed/16_cv_folds_metrics.csv
  - data_processed/16_oof_predictions.csv
  - data_processed/16_roc_curves.html
  - data_processed/16_gain_curves.html

**Próximo:** 17_sentiment_validation → correlacionar probabilidades com retornos (Pearson/Spearman), ANOVA por quantis e lags.
"""
Markdown(md)
